# Scanpy CPU — FULL Workflow (with regress_out + scale)

Matches the official rsc paper (Dicks et al. 2026, Figure 2a).

Change `DATASET` below to switch datasets.

In [ ]:
# === CHANGE THIS TO SWITCH DATASETS ===
DATASET = "pbmc3k"   # or "lung65k" or "mouse_brain_1m"

import json, time, os
import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.neighbors import KNeighborsTransformer

sc.settings.verbosity = 0
print(f"Scanpy: {sc.__version__}")
os.makedirs("write", exist_ok=True)

In [ ]:
with open("benchmark_config.json") as f:
    cfg = json.load(f)
gcfg = cfg["global"]
dcfg = cfg["datasets"][DATASET]
prefix = dcfg["pipeline_prefix"]
timings = {}

out = f"write/{prefix}_scanpy_cpu_full"
print(f"Dataset: {dcfg['name']}")
print(f"Output prefix: {out}")

In [ ]:
t0 = time.time()
adata = sc.read_h5ad(dcfg["canonical_h5ad"])
timings["load"] = time.time() - t0
print(f"Loaded in {timings['load']:.1f}s: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
assert "counts" in adata.layers

## Normalize + log1p

In [ ]:
t0 = time.time()
sc.pp.normalize_total(adata, target_sum=gcfg["target_sum"])
sc.pp.log1p(adata)
timings["normalize_log1p"] = time.time() - t0
print(f"Normalize + log1p: {timings['normalize_log1p']:.1f}s")

## HVG

In [ ]:
t0 = time.time()
sc.pp.highly_variable_genes(
    adata, layer="counts", n_top_genes=dcfg["n_top_genes"],
    flavor="seurat_v3", subset=False, inplace=True,
)
timings["hvg"] = time.time() - t0
print(f"HVG ({adata.var['highly_variable'].sum()} genes): {timings['hvg']:.1f}s")

## Subset to HVGs + Regress out + Scale

**This is what differs from the minimal workflow.**

In [ ]:
adata.raw = adata.copy()
adata = adata[:, adata.var["highly_variable"]].copy()
print(f"Subset to HVGs: {adata.n_obs:,} cells x {adata.n_vars:,} genes")

In [ ]:
# Auto-detect MT column name
mt_col = None
for candidate in ["pct_counts_mt", "pct_counts_MT", "pct_counts_Mt"]:
    if candidate in adata.obs.columns:
        mt_col = candidate
        break

regress_keys = ["total_counts"]
if mt_col:
    regress_keys.append(mt_col)
print(f"Regress out keys: {regress_keys}")

t0 = time.time()
sc.pp.regress_out(adata, keys=regress_keys)
timings["regress_out"] = time.time() - t0
print(f"Regress out: {timings['regress_out']:.1f}s")

In [ ]:
t0 = time.time()
sc.pp.scale(adata, max_value=10)
timings["scale"] = time.time() - t0
print(f"Scale: {timings['scale']:.1f}s")

## PCA (on scaled data)

In [ ]:
t0 = time.time()
sc.pp.pca(
    adata, n_comps=gcfg["pca_n_comps"], svd_solver="arpack",
    random_state=gcfg["random_state"],
)
timings["pca"] = time.time() - t0
print(f"PCA: {timings['pca']:.1f}s")

## Neighbors

In [ ]:
t0 = time.time()
transformer = KNeighborsTransformer(
    n_neighbors=dcfg["n_neighbors"], mode="distance",
    metric=gcfg["neighbor_metric"], algorithm="brute",
)
sc.pp.neighbors(
    adata, n_neighbors=dcfg["n_neighbors"], n_pcs=dcfg["neighbors_n_pcs"],
    use_rep="X_pca", method=gcfg["neighbor_method"],
    transformer=transformer, metric=gcfg["neighbor_metric"],
    random_state=gcfg["random_state"],
)
timings["neighbors"] = time.time() - t0
print(f"Neighbors: {timings['neighbors']:.1f}s")

## Leiden

In [ ]:
t0 = time.time()
leiden_kwargs = {
    "resolution": dcfg["leiden_resolution"],
    "flavor": gcfg["scanpy_leiden_flavor"],
    "random_state": gcfg["random_state"],
    "key_added": "leiden",
}
if gcfg.get("scanpy_leiden_n_iterations") is not None:
    leiden_kwargs["n_iterations"] = gcfg["scanpy_leiden_n_iterations"]
sc.tl.leiden(adata, **leiden_kwargs)
timings["leiden"] = time.time() - t0
n_clusters = adata.obs["leiden"].nunique()
print(f"Leiden ({n_clusters} clusters): {timings['leiden']:.1f}s")

## Checkpoint: save clusters + h5ad before DE

In [ ]:
pd.DataFrame({
    "barcode": adata.obs_names.astype(str),
    "leiden": adata.obs["leiden"].astype(str).values,
}).to_csv(f"{out}_clusters.csv", index=False)
print(f"Checkpoint: clusters saved ({n_clusters} clusters)")

## UMAP

In [ ]:
t0 = time.time()
sc.tl.umap(
    adata, min_dist=gcfg["umap_min_dist"], spread=gcfg["umap_spread"],
    init_pos="spectral", random_state=gcfg["random_state"],
)
timings["umap"] = time.time() - t0
print(f"UMAP: {timings['umap']:.1f}s")

In [ ]:
adata.write_h5ad(f"{out}_pre_de.h5ad", compression="gzip")
print("Checkpoint: h5ad saved (no DE yet)")

## Differential Expression

In [ ]:
t0 = time.time()
sc.tl.rank_genes_groups(
    adata, groupby="leiden", method=gcfg["de_method"],
    corr_method=gcfg["de_corr_method"], use_raw=False, pts=True,
)
timings["de"] = time.time() - t0
print(f"DE: {timings['de']:.1f}s")

## Save all outputs

In [ ]:
markers = sc.get.rank_genes_groups_df(adata, group=None)
markers.to_csv(f"{out}_markers.csv", index=False)
markers_filt = markers[(markers["pvals_adj"] < 0.05) & (markers["logfoldchanges"] > 0.1)]
markers_filt.to_csv(f"{out}_markers_filtered.csv", index=False)

pd.DataFrame(
    adata.obsm["X_umap"], index=adata.obs_names.astype(str),
    columns=["UMAP_1", "UMAP_2"],
).reset_index(names="barcode").to_csv(f"{out}_umap.csv", index=False)

t0 = time.time()
adata.write_h5ad(f"{out}.h5ad", compression="gzip")
timings["save_h5ad"] = time.time() - t0

spec = {
    "pipeline": "scanpy_cpu_full",
    "dataset": dcfg["name"],
    "workflow": "FULL (with regress_out + scale)",
    "scanpy_version": sc.__version__,
    "regress_out_keys": regress_keys,
    "scale_max_value": 10,
    "timings_seconds": timings,
    "results": {
        "n_cells": int(adata.n_obs),
        "n_genes": int(adata.n_vars),
        "n_hvg": dcfg["n_top_genes"],
        "n_clusters": n_clusters,
        "n_marker_rows": len(markers),
        "n_marker_rows_filtered": len(markers_filt),
    },
}
with open(f"{out}_spec.json", "w") as f:
    json.dump(spec, f, indent=2)

print("\n=== Timings ===")
total = 0
for step, t in timings.items():
    print(f"  {step:20s}: {t:8.1f}s")
    total += t
print(f"  {'TOTAL':20s}: {total:8.1f}s ({total/60:.1f} min)")
print(f"\nClusters: {n_clusters}")
print(f"Filtered markers: {len(markers_filt)}")